# Pricing every ad click from SKU economics

**Question:** are we bidding too low on sponsored ads?
**Answer (on this synthetic catalogue, and in the real project it is based on):** the problem isn't the *level* of bids, it's that they're *flat*. What a click is worth varies about 20x across SKUs; what we pay varies about 2x and is uncorrelated with value.

This notebook walks through the method end to end on synthetic data:
1. Simulate a furniture catalogue with realistic economics and flat bids
2. Estimate each SKU's conversion rate with beta-binomial shrinkage
3. Price each click: `MaxCPC = CVR_bayes × contribution per unit`
4. Give every SKU one verdict: bid up / bid down / hold / stop
5. Reallocate the same budget and size the uplift across elasticity assumptions

In [1]:
import sys; sys.path.insert(0, '../src')
import pandas as pd
from bidpricing import simulate_catalogue, price_clicks, dispersion_summary, estimator_error, elasticity_sweep
pd.set_option('display.width', 140)
raw = simulate_catalogue()
raw.head()

,sku,category,price,contribution_per_unit,clicks,orders,spend,cpc,true_cvr
0,SKU-00000,Bar Stools,88.36,-1.34,30,1,3.62,0.121,0.004334
1,SKU-00001,Bookcases,130.17,-2.51,24,0,11.69,0.487,0.006172
2,SKU-00002,Bookcases,403.04,28.24,10,0,1.34,0.134,0.004936
3,SKU-00003,Bar Stools,165.85,-7.97,23,0,4.91,0.214,0.004818
4,SKU-00004,Accent Chairs,272.90,48.21,50,0,9.73,0.195,0.002405


## 1. Shrink thin SKUs toward their category
`CVR_bayes = (orders + k × cohort_cvr) / (clicks + k)` with `k = 300` clicks.

A minimum-order threshold would only score products that already sell, which excludes every growth candidate. Shrinkage scores every SKU, and a SKU with many clicks and no orders is pulled **below** its category, so it isn't credited with orders it didn't get.

Because the data is synthetic we know each SKU's *true* CVR, so we can measure the error directly:

In [2]:
scored = price_clicks(raw, k=300)
(estimator_error(scored) * 100).round(3).rename('mean abs error (pp)')

raw CVR, all SKUs                        0.477
Bayesian CVR, all SKUs                   0.198
raw CVR, thin SKUs (<100 clicks)         0.593
Bayesian CVR, thin SKUs (<100 clicks)    0.214
Name: mean abs error (pp), dtype: float64

## 2. Bids are flat, economics are not

In [3]:
disp = dispersion_summary(scored)
print(f"Spearman(CPC paid, MaxCPC) = {disp.attrs['spearman_cpc_vs_maxcpc']:+.2f}")
disp.round(3)

Spearman(CPC paid, MaxCPC) = -0.02


,p10,median,p90,p90/p10
CPC paid,0.160,0.250,0.389,2.431
MaxCPC affordable,0.030,0.144,0.577,19.534
Contribution $/unit,7.982,42.560,222.984,27.936


![dispersion](../figures/01_dispersion.png)

![scatter](../figures/02_scatter.png)

## 3. One verdict per SKU, with its arithmetic in words
Every row carries a `why` column so any number can be challenged.

In [4]:
print(scored['verdict'].value_counts())
scored.loc[scored.verdict=='BID UP'].sort_values('ratio', ascending=False)[['sku','category','clicks','orders','cpc','max_cpc','ratio','stage','why']].head(5)

verdict
STOP        3936
BID UP      1360
HOLD         485
BID DOWN     219
Name: count, dtype: int64


,sku,category,clicks,orders,cpc,max_cpc,ratio,stage,why
1108,SKU-01108,Wall Beds,554,3,0.105,3.999279,38.088372,1 - proven,"554 clicks, 3 orders (own CVR 0.54%); category..."
963,SKU-00963,Wall Beds,80,0,0.130,3.255445,25.041884,2 - emerging,"80 clicks, 0 orders (own CVR 0.00%); category ..."
1843,SKU-01843,Wall Beds,806,8,0.219,4.578988,20.908619,1 - proven,"806 clicks, 8 orders (own CVR 0.99%); category..."
5504,SKU-05504,Wall Beds,31,1,0.133,2.489583,18.718670,3 - cohort-led,"31 clicks, 1 orders (own CVR 3.23%); category ..."
5958,SKU-05958,Wall Beds,8,0,0.116,1.907009,16.439729,3 - cohort-led,"8 clicks, 0 orders (own CVR 0.00%); category 0..."


## 4. Same budget, re-priced
Volume response to spend is an **assumption** (click elasticity), so the result is reported as a range. Note the `unplaced` column: money freed from unprofitable SKUs that has no profitable home in today's ad pool. That points to the never-advertised catalogue as the next step, not to spending more on the same SKUs.

In [5]:
elasticity_sweep(scored)[['elasticity','budget','freed','unplaced','net_contribution_before','net_contribution_after','uplift']].round(0)

,elasticity,budget,freed,unplaced,net_contribution_before,net_contribution_after,uplift
0,0.0,255442.0,178036.0,101478.0,-11169.0,134754.0,145922.0
1,1.0,255442.0,178036.0,101478.0,-11169.0,177815.0,188984.0
2,1.0,255442.0,178036.0,101478.0,-11169.0,228348.0,239517.0
3,1.0,255442.0,178036.0,101478.0,-11169.0,287641.0,298810.0
4,1.0,255442.0,178036.0,101478.0,-11169.0,357211.0,368379.0


## 5. What I would do next (and did, in the real project)
* **Bounded test before scaling:** raise only *proven* SKUs (own evidence ≥ 50%) to MaxCPC at a retained margin, and measure click elasticity directly. Breakeven elasticity tells you how big the test needs to be.
* **Halo:** halo sales are excluded until incrementality is measured with a holdout. Report the breakeven incrementality instead of assuming it.
* **Floor:** SKUs worth less than the practical CPC floor can't be bid down, only paused. Send their budget to new candidates.